# Park et al. (2020) - Bio-Inspired Edge Detection

**Study**: Park et al., 2020  
**Bio-Inspired Features**: LGN + V1 + Multi-level  
**Architecture**: Multi-level LGN-V1 integration


In [ ]:
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)
import cv2, numpy as np, json
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

OUTPUT_DIR = Path('outputs') / 'Park_2020'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATASET_ROOT = Path('..') / 'datasets' / 'HED_Small'

In [ ]:
def park_2020_detector(img):
    """Park 2020: Multi-level LGN+V1"""
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if len(img.shape) == 3 else img
    
    # Level 1: Fine details
    lgn1 = cv2.GaussianBlur(gray, (0,0), 0.8) - cv2.GaussianBlur(gray, (0,0), 1.6)
    v1_1 = []
    for theta in np.linspace(0, np.pi, 8, endpoint=False):
        kernel = cv2.getGaborKernel((13, 13), 2.5, theta, 8.0, 0.5, 0, ktype=cv2.CV_32F)
        v1_1.append(np.abs(cv2.filter2D(gray, cv2.CV_32F, kernel)))
    level1 = np.abs(lgn1) + np.max(v1_1, axis=0)
    
    # Level 2: Medium details
    lgn2 = cv2.GaussianBlur(gray, (0,0), 1.5) - cv2.GaussianBlur(gray, (0,0), 3.0)
    v1_2 = []
    for theta in np.linspace(0, np.pi, 6, endpoint=False):
        kernel = cv2.getGaborKernel((17, 17), 4.0, theta, 10.0, 0.5, 0, ktype=cv2.CV_32F)
        v1_2.append(np.abs(cv2.filter2D(gray, cv2.CV_32F, kernel)))
    level2 = np.abs(lgn2) + np.max(v1_2, axis=0)
    
    # Fusion
    return cv2.normalize(0.6*level1 + 0.4*level2, None, 0, 1, cv2.NORM_MINMAX)

# Process
img_dir = DATASET_ROOT / 'test' / 'images'
gt_dir = DATASET_ROOT / 'test' / 'edges'
images = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))[:20]

predictions, ground_truths = [], []
for img_path in tqdm(images):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gt_path = gt_dir / img_path.name.replace('.jpg', '.png')
    gt = cv2.imread(str(gt_path), 0).astype(np.float32) / 255.0 if gt_path.exists() else np.zeros(img.shape[:2], dtype=np.float32)
    predictions.append(park_2020_detector(img))
    ground_truths.append(gt)

def compute_metrics(preds, labels):
    t, ois, all_p, all_l = np.linspace(0.05, 0.95, 30), [], [], []
    for p, l in zip(preds, labels):
        l_bin = cv2.dilate((l>0.5).astype(np.float32), np.ones((3,3))).flatten()
        p_smooth = cv2.GaussianBlur(p, (3,3), 0).flatten()
        all_p.append(p_smooth); all_l.append(l_bin)
        ois.append(max([2*np.sum((p_smooth>=th)*l_bin)/(2*np.sum((p_smooth>=th)*l_bin)+np.sum((p_smooth>=th)*(1-l_bin))+np.sum((p_smooth<th)*l_bin)+1e-8) for th in t]))
    fp, fl = np.concatenate(all_p), np.concatenate(all_l)
    ods = max([(2*np.sum((fp>=th)*fl)/(2*np.sum((fp>=th)*fl)+np.sum((fp>=th)*(1-fl))+np.sum((fp<th)*fl)+1e-8), th) for th in t])
    return {'ODS': float(ods[0]), 'ODS_thresh': float(ods[1]), 'OIS': float(np.mean(ois)), 'AP': float(average_precision_score(fl, fp))}

m = compute_metrics(predictions, ground_truths)
print(f"\nPark et al. 2020: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f}")

with open(OUTPUT_DIR / 'park_2020_metrics.json', 'w') as f:
    json.dump({'model': 'Park et al. 2020', 'bio': 'LGN+V1+Multi-level', 'features': 'Hierarchical integration', 'metrics': m}, f, indent=2)
print("✅ Complete!")